# Nemotron Model Reasoning Challenge — LoRA SFT + Submission

This notebook trains a **LoRA adapter** on the **RTX PRO 6000 (96 GB)** GPU and writes `/kaggle/working/submission.zip`. It follows NVIDIA's official submission-demo pattern (`ryanholbrook/nvidia-nemotron-submission-demo`): **internet OFF**, everything from **attached Kaggle resources** — no `pip install`, no `git clone`.

## Required Kaggle UI setup (do this before running)

In the notebook editor sidebar:

1. **Accelerator** -> `GPU RTX PRO 6000`
2. **Internet** -> `OFF`
3. **Add Input** -> attach all three:
   - the **competition** dataset (provides `train.csv`)
   - the model **`metric/nemotron-3-nano-30b-a3b-bf16`**
   - the utility script **`ryanholbrook/nvidia-utility-script`** (provides `nvidia_cutlass_dsl`, required for `mamba_ssm` to import)

Because internet is off there is **no pip install / git clone** — all logic below is inlined into this single notebook.

## Answer format the grader expects

The model is fine-tuned to emit thinking-mode output ending in a boxed answer:

```
<think>...</think>\boxed{answer}
```

LoRA **rank must be <= 32**. The graded answer is whatever is inside `\boxed{}`.

## Config knobs

Leave `SMOKE = True` for a fast sanity run (tiny sample, few steps) to confirm the full pipeline works end-to-end and produces `submission.zip`. **Flip `SMOKE = False` for the real full training run.**

In [ ]:
SMOKE = True  # flip to False for the full training run

MAX_SAMPLES = 200 if SMOKE else None  # cap training rows
MAX_STEPS = 10 if SMOKE else -1  # -1 = no step cap (run full epochs)
MAX_SEQ_LEN = 2048
LORA_RANK = 32  # max allowed is 32
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 1

print(
    f"SMOKE={SMOKE} | MAX_SAMPLES={MAX_SAMPLES} | MAX_STEPS={MAX_STEPS} | "
    f"LORA_RANK={LORA_RANK} | LR={LR} | BATCH_SIZE={BATCH_SIZE} | "
    f"GRAD_ACCUM={GRAD_ACCUM} | EPOCHS={EPOCHS}"
)

In [ ]:
# Load train.csv robustly. The demo path is
#   /kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv
# but we glob recursively so it works regardless of the attached dataset's folder name.
import glob

candidates = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
assert candidates, (
    "train.csv not found under /kaggle/input - attach the competition dataset."
)
TRAIN_CSV = candidates[0]
print("Using:", TRAIN_CSV)

try:
    import polars as pl

    train = pl.read_csv(TRAIN_CSV)
    cols = train.columns
    n_rows = train.height
    USING_POLARS = True
except Exception as e:
    print("polars unavailable, falling back to pandas:", e)
    import pandas as pd

    train = pd.read_csv(TRAIN_CSV)
    cols = list(train.columns)
    n_rows = len(train)
    USING_POLARS = False

print("shape:", (n_rows, len(cols)), "| columns:", cols)

if MAX_SAMPLES is not None:
    train = train.head(MAX_SAMPLES)
    print(f"capped to MAX_SAMPLES={MAX_SAMPLES} rows")

In [ ]:
# Make the utility-script's vendored packages importable so mamba_ssm can load.
# (Path comes from ryanholbrook/nvidia-utility-script; wrapped in try/except in case the
# attached utility-script's layout differs.)
import site

CUTLASS_SITEDIR = (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/"
    "nvidia_cutlass_dsl/python_packages/"
)
try:
    site.addsitedir(CUTLASS_SITEDIR)
    print("added sitedir:", CUTLASS_SITEDIR)
except Exception as e:
    print("could not add cutlass sitedir (check the attached utility-script path):", e)

import kagglehub
import mamba_ssm  # noqa: F401  (import side effect: registers Mamba CUDA kernels)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print("MODEL_PATH:", MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded")

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Thinking-mode SFT dataset

Each training example is rendered through the tokenizer's chat template as a `user` prompt + an `assistant` reply in the grader's expected format: an (empty, for SFT) `<think>` block followed by `\boxed{answer}`. Answers are cast to `str` so leading zeros are preserved.

In [ ]:
def format_target(answer: str) -> str:
    return f"<think>\n\n</think>\n\n\\boxed{{{answer}}}"


def to_sft_text(prompt: str, answer: str) -> str:
    msgs = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": format_target(answer)},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)


# Pull prompt/answer columns out as plain python lists (polars or pandas).
if USING_POLARS:
    prompts = train["prompt"].to_list()
    answers = [str(a) for a in train["answer"].to_list()]
else:
    prompts = train["prompt"].tolist()
    answers = [str(a) for a in train["answer"].tolist()]

texts = [to_sft_text(p, a) for p, a in zip(prompts, answers, strict=False)]
print(f"built {len(texts)} SFT examples\n")
print("=== example SFT text ===")
print(texts[0])

In [ ]:
# Minimal, offline-safe causal-LM SFT loop (no TRL dependency).
import torch
from torch.utils.data import DataLoader, Dataset

PAD_ID = tokenizer.pad_token_id


class SFTDataset(Dataset):
    def __init__(self, texts):
        self.ids = [
            tokenizer(t, truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]
            for t in texts
        ]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i]


def collate(batch):
    maxlen = max(len(x) for x in batch)
    input_ids, attn, labels = [], [], []
    for x in batch:
        pad = maxlen - len(x)
        input_ids.append(x + [PAD_ID] * pad)
        attn.append([1] * len(x) + [0] * pad)
        labels.append(x + [-100] * pad)  # pad positions ignored in loss
    return (
        torch.tensor(input_ids),
        torch.tensor(attn),
        torch.tensor(labels),
    )


loader = DataLoader(
    SFTDataset(texts), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)

step = 0
stop = False
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    for micro, (input_ids, attn, labels) in enumerate(loader):
        input_ids = input_ids.to(model.device)
        attn = attn.to(model.device)
        labels = labels.to(model.device)

        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        loss = out.loss / GRAD_ACCUM
        loss.backward()

        if (micro + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            print(f"epoch {epoch} | step {step} | loss {out.loss.item():.4f}")
            if MAX_STEPS > 0 and step >= MAX_STEPS:
                stop = True
                break
    if stop:
        break

print("training done; optimizer steps:", step)

## Save + package the submission

Saving the PEFT model writes only the small adapter files (`adapter_config.json`, `adapter_model.safetensors`) to `/kaggle/working`. We zip exactly those into `submission.zip` (naming them explicitly so we never sweep in the input dataset).

In [ ]:
import os
import subprocess

model.save_pretrained("/kaggle/working")

os.chdir("/kaggle/working")
subprocess.run(
    "zip -m submission.zip adapter_config.json adapter_model.safetensors",
    shell=True,
    check=True,
)
print("submission.zip ready:", os.path.exists("/kaggle/working/submission.zip"))

## Submit

Submit `/kaggle/working/submission.zip` to the competition:

- Click the competition **Submit** button on this notebook's output, **or**
- from a machine with the Kaggle CLI:

```bash
kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f submission.zip \
  -m "LoRA SFT thinking-mode r=32"
```

The adapter is graded **server-side via vLLM** on held-out prompts. Requirements: LoRA **rank <= 32**, and answers emitted inside `\boxed{}` after a `<think>...</think>` block.

The repo's `src/` and `scripts/` directories hold the same training + SFT-formatting logic for the local / A100 path (see `scripts/train_a100.sh`).